In [1]:
import json
from pathlib import Path

import torch
from torch.nn import functional as F

import config as cfg
from config import *

from skywork_tokenizer import SkyworkTokenizerAPI
from skywork_o1_prm_inference.model_utils.prm_model import PRM_MODEL

DATA_PATH = Path("phase2_train.jsonl")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

/home/eecs/hengyang/miniconda3/envs/reasoning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
def load_prm800k(jsonl_path, max_samples=None):
    """
    Returns a list of dicts:
    {
        "idx": int,
        "question": str,
        "answer": str or list,
        "raw": original_json_obj
    }
    """
    examples = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if max_samples is not None and idx >= max_samples:
                break
            line = line.strip()
            if not line:
                continue
            data = json.loads(line)
            q = data["question"]["problem"]
            a = data["question"]["pre_generated_steps"]
            examples.append({
                "idx": idx,
                "question": q,
                "answer": a,
                "raw": data,
            })
    return examples

examples = load_prm800k(DATA_PATH)
print(f"Loaded {len(examples)} examples from {DATA_PATH}")

Loaded 97782 examples from phase2_train.jsonl


In [3]:
def search_examples(examples, phrase, max_results=10):
    phrase_lower = phrase.lower()
    matches = []
    for ex in examples:
        q = ex["question"]
        a = ex["answer"]
        text = q
        # pre_generated_steps might be a list or a string
        if isinstance(a, list):
            text += " " + " ".join(map(str, a))
        else:
            text += " " + str(a)
        if phrase_lower in text.lower():
            matches.append(ex)
        if len(matches) >= max_results:
            break
    return matches

# Interactive search
phrase = input("Enter a search phrase: ").strip()
matches = search_examples(examples, phrase, max_results=5)

if not matches:
    print("No matches found.")
else:
    print(f"Found {len(matches)} matches:")
    for i, ex in enumerate(matches):
        print("=" * 60)
        print(f"[match {i}] global_idx={ex['idx']}")
        print("QUESTION:")
        print(ex["question"])
        print("\nANSWER (pre_generated_steps):")
        print(ex["answer"])

Found 1 matches:
[match 0] global_idx=0
QUESTION:
The first four terms in an arithmetic sequence are $x+y$, $x-y$, $xy$, and $x/y$, in that order. What is the fifth term? Express your answer as a common fraction.

ANSWER (pre_generated_steps):
['To find the fifth term, I need to identify the common difference of the arithmetic sequence and add it to the fourth term.', 'The common difference is the same for any consecutive pair of terms, so I can use any of them to find it.', 'For example, using the first and second terms, I can write $x-y = x+y + d$, where $d$ is the common difference.', 'Solving for $d$, I get $d = -2y$.', 'Using another pair of terms, such as the second and third, I can check if this value of $d$ is consistent.', 'I have $xy = x-y + d$, so substituting $d = -2y$, I get $xy = x-y - 2y$.', 'Simplifying, I get $xy = x - 3y$.', 'This seems like a reasonable equation, so I will assume that $d = -2y$ is correct.', 'Now, to find the fifth term, I need to add $d$ to the four

In [4]:
if not matches:
    chosen = None
else:
    sel = input(f"Enter match index to evaluate (0–{len(matches)-1}): ").strip()
    try:
        sel = int(sel)
        chosen = matches[sel]
    except Exception:
        print("Invalid choice.")
        chosen = None

if chosen is not None:
    print("\nYou selected:")
    print("=" * 60)
    print("QUESTION:")
    print(chosen["question"])
    print("\nANSWER (pre_generated_steps):")
    print(chosen["answer"])


You selected:
QUESTION:
The first four terms in an arithmetic sequence are $x+y$, $x-y$, $xy$, and $x/y$, in that order. What is the fifth term? Express your answer as a common fraction.

ANSWER (pre_generated_steps):
['To find the fifth term, I need to identify the common difference of the arithmetic sequence and add it to the fourth term.', 'The common difference is the same for any consecutive pair of terms, so I can use any of them to find it.', 'For example, using the first and second terms, I can write $x-y = x+y + d$, where $d$ is the common difference.', 'Solving for $d$, I get $d = -2y$.', 'Using another pair of terms, such as the second and third, I can check if this value of $d$ is consistent.', 'I have $xy = x-y + d$, so substituting $d = -2y$, I get $xy = x-y - 2y$.', 'Simplifying, I get $xy = x - 3y$.', 'This seems like a reasonable equation, so I will assume that $d = -2y$ is correct.', 'Now, to find the fifth term, I need to add $d$ to the fourth term.', 'The fourth te

In [5]:
skywork_tokenizer_api = SkyworkTokenizerAPI(
    cfg.SKYWORK_MODEL_NAME, cfg.STEP_TOKEN
)

reward_model = PRM_MODEL.from_pretrained(cfg.SKYWORK_MODEL_NAME)
reward_model.to(device)
reward_model.eval()

print("PRM model loaded.")

/rscratch/hengyang/prm-attack/skywork_o1_prm_inference/model_utils/modeling_base.py:264: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = loading_func(filename if 

PRM model loaded.


In [6]:
def evaluate_example(question, answer):
    """
    question: str
    answer: str or list (as in PRM800k)
    Returns (avg_reward_prob, nll)
    """
    # SkyworkTokenizerAPI expects lists of questions/answers
    tokenized = skywork_tokenizer_api.prepare_steps([question], [answer])
    tokenized = tokenized.to(device)

    with torch.no_grad():
        # Standard call: use input_ids directly, no adversarial prefix
        output = reward_model(**tokenized, return_probs=True)

    # Following your training loop: output[2] contains probabilities
    reward_probs = output[2]  # shape: [batch, seq]
    reward_flags = tokenized.data["reward_flags"].bool()

    # For single example, that's fine; just mask and reduce
    probs_on_rewards = reward_probs[reward_flags]
    avg_reward_prob = probs_on_rewards.mean().item()
    nll = (-probs_on_rewards.log()).mean().item()

    return avg_reward_prob, nll

In [7]:
if chosen is None:
    print("No example selected.")
else:
    q = chosen["question"]
    a = chosen["answer"]
    avg_p, nll = evaluate_example(q, a)

    print("\n=== PRM evaluation ===")
    print(f"Average reward probability over reward_flags: {avg_p:.4f}")
    print(f"NLL over reward_flags:                        {nll:.4f}")


=== PRM evaluation ===
Average reward probability over reward_flags: 0.1211
NLL over reward_flags:                        2.2726
